# Synthetic Controlled Datasets

This notebook runs the four artificial datasets from TODO point 5: correlated helper columns, high-cardinality categoricals, rare categories, and a sensitive identifier column.


## Setup

The synthetic datasets are deterministic and are written to `experiments/data/processed/` before exploration and experiments run.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "experiments").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from experiments.datasets import DATASET_CONFIGS
from experiments.exploration import column_distribution_summary, plot_column_distributions, plot_pairwise_features
from experiments.numeric_projection import plot_numeric_projection_triptych
from experiments.synthetic_data import SYNTHETIC_DATASETS, materialize_synthetic_datasets
from experiments.workflow import dataset_profile, load_dataset, run_configured_dataset_experiment, working_dataframe

SYNTHETIC_KEYS = [spec.key for spec in SYNTHETIC_DATASETS]
RESULTS_DIR = ROOT / "experiments" / "results"


In [ ]:
materialize_synthetic_datasets(ROOT / "experiments" / "data" / "processed")


## Dataset Definitions


In [ ]:
pd.DataFrame([spec.__dict__ for spec in SYNTHETIC_DATASETS])


## Exploratory Data Analysis

Inspect one controlled dataset at a time by changing `EDA_KEY`. These cells run before the experiment workflow.


In [ ]:
EDA_KEY = "synthetic_rare_categories"
EDA_CONFIG = DATASET_CONFIGS[EDA_KEY]
dataframe = load_dataset(EDA_CONFIG, root=ROOT)
work = working_dataframe(dataframe, EDA_CONFIG)
profile = dataset_profile(dataframe)
EDA_KEY, dataframe.shape


In [ ]:
profile


In [ ]:
column_distribution_summary(dataframe)


In [ ]:
plot_column_distributions(dataframe, title=EDA_CONFIG.title)


In [ ]:
plot_pairwise_features(work, target_column=EDA_CONFIG.target_column, title=EDA_CONFIG.title)


## Run All Controlled Experiments

The sampler/baseline workflows run after the exploratory cells.


In [ ]:
results = {
    key: run_configured_dataset_experiment(DATASET_CONFIGS[key], root=ROOT, results_dir=RESULTS_DIR)
    for key in SYNTHETIC_KEYS
}
list(results)


## Numeric Projection Of Generated Data

In [ ]:
projection_result = results[EDA_KEY]
projection_path = ROOT / "experiments" / "figures" / f"{EDA_KEY}_numeric_projection.pdf"
_ = plot_numeric_projection_triptych(
    projection_result.starter_run.sampler,
    projection_result.working_dataframe,
    projection_result.starter_run.generated,
    title=EDA_CONFIG.title,
    reducer="umap",
    random_state=EDA_CONFIG.random_state,
    output_path=projection_path,
)
projection_path

## Experiment Profiles


In [ ]:
{key: result.dataframe.shape for key, result in results.items()}


In [ ]:
results["synthetic_rare_categories"].profile


## Comparison Summaries


In [ ]:
pd.concat([result.comparison for result in results.values()], ignore_index=True)
